# HaluRISC — Full Training Pipeline (Colab)

Runs the complete Version A experiment protocol on Colab GPU:

1. HaluEval download + prep (70/15/15 stratified splits)
2. Full feature extraction (length, lexical, entity/NER, **NLI**, numeric, hedging, semantic)
3. XGBoost tuning (30 iters, 5-fold CV) + baselines + 3-seed protocol
4. Platt vs isotonic calibration, ECE/Brier, McNemar, bootstrap CIs, ablations
5. SHAP global + local explanations
6. RAGTruth zero-shot external validation
7. Zip artifacts + feature matrix to **Google Drive** (persists) for download

**Before starting:** have `colab/halurisc_src.zip` from the repo ready. Cell 3 opens a file-picker to select it from your laptop (no Drive upload needed). Alternatively, upload it once to `MyDrive/HaluRISC/halurisc_src.zip` and it will be picked up automatically.

**Runtime:** enable GPU (Runtime > Change runtime type > T4). Total wall time ≈ 10–20 min.

In [ ]:
# 1) Mount Google Drive (artifacts persist here across sessions)
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/HaluRISC'
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Drive mounted at', DRIVE_DIR)

In [ ]:
# 2) Environment check: GPU must be enabled
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
if not torch.cuda.is_available():
    print('!! No GPU detected - enable GPU in Runtime > Change runtime type')
    raise SystemExit(1)

In [ ]:
# 3) Get the HaluRISC source. PREFERRED: browser file-picker (select colab/halurisc_src.zip from your laptop).
#    Fallback: if you already put halurisc_src.zip inside Drive/HaluRISC/, it is used automatically.
import zipfile, io, os
from google.colab import files

ROOT = '/content/HaluRISC'
os.makedirs(ROOT, exist_ok=True)

if not os.path.exists(os.path.join(ROOT, 'src')):
    src_zip = None
    drive_zip = os.path.join(DRIVE_DIR, 'halurisc_src.zip')
    if os.path.exists(drive_zip):
        src_zip = open(drive_zip, 'rb')
        print('Found halurisc_src.zip on Drive - extracting...')
        with zipfile.ZipFile(src_zip) as z:
            z.extractall(ROOT)
    else:
        print('Upload halurisc_src.zip (repo/colab/halurisc_src.zip) - use the file picker:')
        uploaded = files.upload()
        for name, content in uploaded.items():
            with zipfile.ZipFile(io.BytesIO(content)) as z:
                z.extractall(ROOT)
    print('Extracted to', ROOT)
print('src present:', os.path.exists(os.path.join(ROOT, 'src')))
%cd {ROOT}

In [ ]:
# 4) Install pinned dependencies (Colab keeps its own torch)
!pip install -q -r colab/requirements-colab.txt
!python -m spacy download en_core_web_sm -q
print('deps OK')

In [ ]:
# 5) HaluEval download + prepare (downloads ~6MB from GitHub)
!python src/data/download.py
!python src/data/prepare.py

In [ ]:
# 6) Full feature extraction (7 groups, ~40K NLI pairs on GPU; ~5-10 min)
# Default NLI model: cross-encoder/nli-deberta-v3-base. Fallback: --nli-model cross-encoder/nli-MiniLM2-L6
!python src/features/extract_features.py

In [ ]:
# 7) Full experiment protocol: tuning, baselines, 3 seeds, calibration, stats, ablations (~10-20 min)
!python src/models/train_pipeline.py

In [ ]:
# 8) SHAP explanations + calibration/ROC/PR figures
!python src/explain/shap_analysis.py

In [ ]:
# 9) RAGTruth zero-shot external validation (downloads from HuggingFace)
!python src/data/download_ragtruth.py
!python src/models/eval_ragtruth.py

In [ ]:
# 10) Show the headline results
import json, pandas as pd
res = json.load(open('artifacts/results/final_results.json'))
rows = {k: v for k, v in res.items() if isinstance(v, dict) and 'f1' in v and 'auroc' in v}
df = pd.DataFrame(rows).T[['precision','recall','f1','auroc','pr_auc','mcc']].round(4)
print('MODEL COMPARISON (test set, mean over seeds 42/123/456)')
print(df.to_string())
print('\nCALIBRATION:', json.dumps(res['calibration'], indent=2))
print('\nABLATION:')
print(pd.DataFrame(res['ablation']).to_string(index=False))

In [ ]:
# 11) Package artifacts to Drive (persists across sessions) and offer a download link
import os, shutil, zipfile
from datetime import date
from google.colab import files

stamp = date.today().isoformat()
zip_path = f'{DRIVE_DIR}/halurisc_artifacts_{stamp}.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, fnames in os.walk('artifacts'):
        for fn in fnames:
            p = os.path.join(root, fn)
            z.write(p, os.path.relpath(p, '.'))
    for fn in ['data/processed/features_full.parquet', 'data/processed/qa_clean.parquet']:
        if os.path.exists(fn):
            z.write(fn)
print('Saved:', zip_path, f'({os.path.getsize(zip_path)/1e6:.1f} MB)')
print()
print('NEXT: download the zip from your Drive, unzip at the repo root of your laptop.')
print('The API (uvicorn) and web dashboard will then load the real artifacts.')
files.download(zip_path)